In [10]:
import numpy as np
import tempfile
import torch
import torch.nn as nn
import gradio as gr


In [11]:
from architecture import ConvBlock,Encoder,Encodings,CrossAttention,SelfAttention,FNN,BLOCK,Decoder,Model


In [12]:
model=Model(ConvBlock,Encoder,Encodings,CrossAttention,SelfAttention,FNN,BLOCK,Decoder)
model.float()

Model(
  (encoder_optical): Encoder(
    (ConvBlock1): ConvBlock(
      (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (instancenorm1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (instancenorm2): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (activation): LeakyReLU(negative_slope=0.2)
    )
    (ConvBlock2): ConvBlock(
      (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (instancenorm1): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (instancenorm2): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (activation): LeakyReLU(negative_slope=0.2)
    )
    (ConvBlock3): ConvBlock(
      (c

In [15]:
model.load_state_dict(torch.load('MODEL',map_location='cpu'))

<All keys matched successfully>

In [16]:
def predict(opt_path, sar_path):
    opt = np.load(opt_path).astype(np.float32)    # (3,256,256), [0,1]
    sar = np.load(sar_path).astype(np.float32)    # (2,256,256), [-1,1]
    if opt.shape != (3, 256, 256) or sar.shape != (2, 256, 256):
        raise gr.Error(f"Bad shapes: optical {opt.shape}, sar {sar.shape}")

    Opt=torch.from_numpy(opt).unsqueeze(0)
    Sar=torch.from_numpy(sar).unsqueeze(0)
    
    with torch.no_grad():
        out = model(Opt,Sar)[0].clamp(0,1).numpy()                 # (3,256,256)

    f = tempfile.NamedTemporaryFile(suffix=".npy", delete=False)
    np.save(f.name, out)
    preview = (out.transpose(1, 2, 0) * 255).astype(np.uint8)
    return f.name, preview
    
    

In [ ]:
demo = gr.Interface(
    fn=predict,
    inputs=[gr.File(label="optical.npy", file_types=[".npy"]),
            gr.File(label="sar.npy", file_types=[".npy"])],
    outputs=[gr.File(label="output.npy"), gr.Image(label="preview")],
    api_name="predict",
)
demo.launch()